# A ReAct agent end to end

Booster LEAP, Agentic AI, topic 3. Run the whole agent of the lesson, change it, and (optionally) connect it to a real model with your own key.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: the agent with a scripted model

The model here is a function that hands out prepared replies, so every run is the same and nothing is paid for. The first cell holds the tools of the support desk (lessons of topic 2) with their owner and status checks. Run it first. **Run it again whenever you want the orders back in their starting state:** a refund changes the data.

In [ ]:
import json
import re

ORDERS = {
    "A1001": {"owner": "C1", "status": "delivered", "total": 799, "refunded": 0},
    "A1002": {"owner": "C1", "status": "delivered", "total": 450, "refunded": 200},
    "A1003": {"owner": "C2", "status": "shipped", "total": 299, "refunded": 0},
}
SESSION = {"customer": "C1"}
POLICY = {"refund": "Refunds are possible only for delivered orders. A refund is at most 500 rupees and at most what is left on the order."}

def lookup_order(order_id):
    o = ORDERS.get(order_id)
    if o is None or o["owner"] != SESSION["customer"]:
        raise ValueError("no such order for this customer")
    return f"order {order_id}: {o['status']}, total {o['total']}, refunded {o['refunded']}"

def search_policy(query):
    for key, text in POLICY.items():
        if key in query.lower():
            return text
    return "no policy found"

def issue_refund(order_id, amount, reason):
    o = ORDERS.get(order_id)
    if o is None or o["owner"] != SESSION["customer"]:
        raise ValueError("no such order for this customer")
    if o["status"] != "delivered":
        raise ValueError("the order is not delivered")
    if not isinstance(amount, int) or not 1 <= amount <= min(500, o["total"] - o["refunded"]):
        raise ValueError(f"amount must be a whole number from 1 to {min(500, o['total'] - o['refunded'])}")
    o["refunded"] += amount
    return f"refunded {amount} rupees on {order_id} ({reason})"

FUNCS = {"lookup_order": lookup_order, "search_policy": search_policy, "issue_refund": issue_refund}
NEEDS = {"lookup_order": ["order_id"], "search_policy": ["query"], "issue_refund": ["order_id", "amount", "reason"]}

The next cell holds the pieces of this topic: the parser of ReAct replies (lesson 1), the plan check (lesson 3), the repeat detector (lesson 4) and the check of the final answer (lesson 5). It prints nothing.

In [ ]:
def parse(reply):
    """Lesson 1: turn the model's text into ('final', text), ('action', name, args) or ('error', message for the model)."""
    final = re.search(r"Final Answer:\s*(.+)", reply, re.S)
    if final:
        return ("final", final.group(1).strip())
    name = re.search(r"Action:\s*(\w+)", reply)
    if not name:
        return ("error", "no Action line: write 'Action: <tool>' and 'Action Input: <json>', or 'Final Answer: <text>'")
    if name.group(1) not in NEEDS:
        return ("error", f"unknown tool '{name.group(1)}'; the tools are {', '.join(NEEDS)}")
    raw = re.search(r"Action Input:\s*(\{.*\})", reply, re.S)
    if not raw:
        return ("error", "no Action Input line with a JSON object")
    try:
        args = json.loads(raw.group(1))
    except json.JSONDecodeError:
        return ("error", "Action Input is not valid JSON")
    missing = [k for k in NEEDS[name.group(1)] if k not in args]
    if missing:
        return ("error", f"missing arguments {missing}")
    return ("action", name.group(1), args)

def check_plan(reply, max_steps=4):
    """Lesson 3: a plan line is checked before anything runs. Returns a message for the model, or None if the plan is fine."""
    names = [n.strip() for n in reply.split("Plan:", 1)[1].split(",")]
    unknown = [n for n in names if n not in NEEDS]
    if unknown:
        return f"plan rejected: unknown tool {unknown[0]}"
    if len(names) > max_steps:
        return f"plan rejected: {len(names)} steps, at most {max_steps}"
    return None

def detect(trace):
    """Lesson 4: the same call twice in a row, or three identical errors."""
    acts = [a for a, _ in trace]
    obs = [o for _, o in trace]
    if len(acts) >= 2 and acts[-1] == acts[-2]:
        return "same call twice in a row"
    if len(obs) >= 3 and len(set(obs[-3:])) == 1 and obs[-1].startswith("ERROR"):
        return "three identical errors"
    return None

def check_answer(answer, refunds):
    """Lesson 5: a real check of the final answer against what the tools really did."""
    problems = []
    for amount in re.findall(r"refunded (\d+) rupees", answer):
        if int(amount) not in refunds:
            problems.append(f"the answer says {amount} rupees were refunded, but the tools refunded {refunds or 'nothing'}")
    return problems

Now the loop itself, with a step limit and a token budget, and the stand-in model. It prints nothing; it only defines `run`, `stop` and `scripted`.

In [ ]:
def run(question, model, max_steps=8, token_budget=600, max_revisions=1):
    """model(messages) -> the next reply as text. messages is a list of (role, text)."""
    print("user:", question)
    messages = [("user", question)]
    trace, refunds, revisions = [], [], 0
    for step in range(1, max_steps + 1):
        if sum(len(t) for _, t in messages) // 4 > token_budget:             # lesson 4: a rough estimate of the tokens
            return stop("token budget", trace)
        reply = model(messages)
        messages.append(("assistant", reply))
        if reply.startswith("Plan:"):
            problem = check_plan(reply)
            print(f"  {step} plan {reply[5:].strip()} -> {problem or 'accepted'}")
            messages.append(("user", "Observation: " + (problem or "plan accepted, go on")))
            continue
        kind, *rest = parse(reply)
        if kind == "error":
            print(f"  {step} bad reply -> {rest[0]}")
            messages.append(("user", "Observation: ERROR: " + rest[0]))
            continue
        if kind == "final":
            problems = check_answer(rest[0], refunds)
            if problems and revisions < max_revisions:
                revisions += 1
                print(f"  {step} final answer rejected by the check -> {problems[0]}")
                messages.append(("user", "Observation: ERROR: " + problems[0] + ". Write the final answer again."))
                continue
            print(f"  {step} final answer: {rest[0]}")
            return "answered"
        name, args = rest
        try:
            observation = str(FUNCS[name](**args))
        except Exception as e:
            observation = f"ERROR: {e}"
        action = f"{name}({json.dumps(args)})"
        print(f"  {step} {action} -> {observation}")
        if name == "issue_refund" and not observation.startswith("ERROR"):
            refunds.append(args["amount"])
        trace.append((action, observation))
        messages.append(("user", "Observation: " + observation))
        reason = detect(trace)
        if reason:
            return stop(reason, trace)
    return stop(f"step limit of {max_steps} reached", trace)

def stop(reason, trace):
    last = trace[-1][1] if trace else "nothing yet"
    print(f"  stopped ({reason}). Last result: {last}. A person should look at this case.")
    return "stopped"

def scripted(replies):
    """A stand-in model: it hands out prepared replies. A real model would take its place."""
    replies = list(replies)
    return lambda messages: replies.pop(0)

Three runs: a lookup with a plan; a refund with a rejected plan, an unknown tool and a wrong final answer; and a loop that is stopped. Read each trace line: step, action, result.

In [ ]:
def act(name, args, thought="I need this fact."):
    return f"Thought: {thought}\nAction: {name}\nAction Input: {json.dumps(args)}"

run("Where is order A1002?", scripted([
    "Plan: lookup_order",
    act("lookup_order", {"order_id": "A1002"}),
    "Final Answer: Order A1002 was delivered. 200 of 450 rupees are already refunded.",
]))
run("Please refund what is left on A1002, it arrived late.", scripted([
    "Plan: lookup_order, delete_order",
    "Plan: lookup_order, issue_refund",
    act("lookup_order", {"order_id": "A1002"}),
    act("refund_order", {"order_id": "A1002"}),
    act("issue_refund", {"order_id": "A1002", "amount": 250, "reason": "late"}),
    "Final Answer: I refunded 450 rupees on order A1002.",
    "Final Answer: I refunded 250 rupees on order A1002 for the late delivery.",
]))
run("Where is order A1009?", scripted([
    "Thought: look it up.\nAction: lookup_order\nAction Input: {order_id: A1009}",
    act("lookup_order", {"order_id": "A1009"}),
    act("lookup_order", {"order_id": "A1009"}),
]))

user: Where is order A1002?
  1 plan lookup_order -> accepted
  2 lookup_order({"order_id": "A1002"}) -> order A1002: delivered, total 450, refunded 200
  3 final answer: Order A1002 was delivered. 200 of 450 rupees are already refunded.
user: Please refund what is left on A1002, it arrived late.
  1 plan lookup_order, delete_order -> plan rejected: unknown tool delete_order
  2 plan lookup_order, issue_refund -> accepted
  3 lookup_order({"order_id": "A1002"}) -> order A1002: delivered, total 450, refunded 200
  4 bad reply -> unknown tool 'refund_order'; the tools are lookup_order, search_policy, issue_refund
  5 issue_refund({"order_id": "A1002", "amount": 250, "reason": "late"}) -> refunded 250 rupees on A1002 (late)
  6 final answer rejected by the check -> the answer says 450 rupees were refunded, but the tools refunded [250]
  7 final answer: I refunded 250 rupees on order A1002 for the late delivery.
user: Where is order A1009?
  1 bad reply -> Action Input is not valid JSON
  

## Experiments (stage 1)

Change one thing at a time, predict the trace, then run the cell above again (run the tools cell first to reset the orders).

1. Set `max_steps` in `run` to 3. Which runs end with "step limit of 3 reached"?
2. In run 2 remove the first plan line. How many steps does the run need now?
3. In run 2 replace the second final answer by the first one. What happens when the check has used its one revision? How would you change `run` so that the answer is flagged instead of accepted?
4. Set `token_budget` to 50. At which step does the first run stop, and what does it report?
5. Add a fourth run in which the model repeats `search_policy` with the same query three times. Which brake fires, and at which step?
6. Add a second check to `check_answer`: a final answer must not say a refund was made when `refunds` is empty. Which run would catch it?

## Stage 2 (optional): a real model

Everything above ran with prepared replies. This stage uses a real model with **your own key**. Read before you run:

- It sends your question and the observations to the provider and costs a small amount of money. The step limit of 8 and the token budget keep one question cheap, but you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- Real models behave differently from a script. Some write the format perfectly, some add extra text, some answer without calling a tool. The parser and the checks handle that, and the trace shows what happened. Results vary from run to run.

In [ ]:
# Optional stage 2: the same run() with a real model. This sends the questions and the observations to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip -q install openai
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"
SYSTEM = ("You are a support desk assistant. Answer with exactly one of these forms.\n"
          "Thought: <one sentence>\nAction: <lookup_order, search_policy or issue_refund>\nAction Input: <a JSON object>\n"
          "or\nFinal Answer: <text>\nAfter each action you receive an Observation.")


def real_model(messages):
    chat = [{"role": "system", "content": SYSTEM}] + [{"role": role, "content": text} for role, text in messages]
    return client.chat.completions.create(model=MODEL, messages=chat).choices[0].message.content

In [ ]:
run("Where is order A1001?", real_model)

In [ ]:
run("Please refund 100 rupees on A1001, it arrived damaged.", real_model)

In [ ]:
run("Where is order A1003?", real_model)          # this order belongs to another customer

## What to look at

1. How many steps did the real model need for the first question? Compare with the scripted run.
2. Did the model ever write a reply that the parser rejected? Which message did it get, and did it repair the reply?
3. For the refund: did the final answer pass the check at once? Which amount did the model choose?
4. For the third question: what did the tool return, and what did the model tell the user? Did it make up an answer?
5. Write down one case where the real model and the scripted model differ, and which part of the program protected you.